# Class 12 - Data Cleaning
### Module 3 . Week 4 . Sunday

Real data is never clean. Hospital records have missing blood pressure
readings. Ages export as "35 yrs" instead of 35. Duplicate patient IDs sneak
in from system migrations. Gender gets recorded as "M", "Male", "male", and
"m" in the same column.

Data cleaning isn't a chore before the "real" analysis - it **is** the real
analysis. A model trained on dirty data produces dirty conclusions, no matter
how sophisticated the model.

Today's dataset is a small synthetic patient log, deliberately messy in
every way real clinical data tends to be messy.

* **`patients.csv` (Primary)**: A 16-row synthetic clinical log containing missing vitals, text-polluted age fields (`"34 yrs"`, `"51yo"`), dirty whitespace, mixed gender casing (`"Male"`, `"f"`, `"female "`), duplicate patient visits, and a sentinel outlier (`999.0` systolic BP).

| Topic | Key Methods / Functions | Parameters & Flags Learned | Practical Purpose |
| --- | --- | --- | --- |
| **Null Auditing** | `.isnull()`, `.isna()`, `.sum()`, `.mean()`, `.any()` | `axis=0` (column-wise), `axis=1` (row-wise) | Quantify missing values as counts or percentages across rows and columns. |
| **Row Dropping** | `.dropna()` | `how='any'` / `'all'`, `subset=['col']`, `thresh=N` | Remove rows with missing values aggressively or conditionally based on non-null thresholds. |
| **Imputation** | `.fillna()` | `value=...`, `method='ffill'` / `'bfill'` | Replace `NaN` values with constants, aggregated metrics (mean/median), or time-series fills. |
| **Deduplication** | `.duplicated()`, `.drop_duplicates()` | `subset=['col']`, `keep='first'` / `'last'` / `False` | Detect and drop exact duplicate rows or group-specific duplicates based on primary keys. |
| **Type Coercion** | `pd.to_numeric()`, `pd.to_datetime()`, `.astype()` | `errors='coerce'` / `'raise'`, `format=...` | Safely force mixed or text columns into numeric or datetime types without breaking execution. |
| **String Normalization** | `.str.strip()`, `.str.split()`, `.str.join()`, `.str.lower()`, `.str.title()` | Delimiters, whitespace joins | Clean leading, trailing, and internal multi-space padding, and unify text casing. |
| **Pattern Extraction** | `.str.extract()` | `r'(\d+)'` (regex capture groups), `expand=False` | Isolate digits or specific text patterns out of messy string values. |
| **Sentinel Masking** | `.where()` | Condition mask (e.g., `df['bp_sys'] < 300`) | Blank out impossible sensor readings (e.g., `999.0`) to `NaN` prior to statistical computation. |
| **Header Cleanup** | `df.columns.str...` | `.str.strip().str.lower().str.replace()` | Programmatically standardize dirty column headers into uniform `snake_case`. |

## Learning Objectives

- Quantify missing data with `.isnull()` before deciding how to handle it
- Choose between `.dropna()` and `.fillna()` for a given situation
- Detect and remove duplicates with intent, not by default
- Coerce broken types safely with `errors='coerce'`
- Clean text columns with vectorised `.str` methods
- Produce a before/after validation report

## 1. Necessary Import

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "./datasets"

raw = pd.read_csv(r"C:\Users\PMLS\Desktop\hands-on-data-analytics-python\notebooks\03_Module3_Pandas\week_04\datasets\patients.csv")
raw

,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
1,P002,Sara Khan,28 years,female,118.0,2024-01-11,36.7
2,P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
3,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
4,P004,Zara Malik,45 yr old,FEMALE,132.0,2024-01-13,38.2
5,P005,hassan ALI,67,male,142.0,2024-01-14,34.3
6,P006,Nida Butt,39 years,F,NaN,2024-01-15,38.1
7,P007,USMAN sheikh,NaN,Female,999.0,2024-01-16,35.2
8,P008,Ayesha Nawaz,31 yrs,female,128.0,2024-01-17,36.7
9,P003,BILAL ahmed,51yo,M,158.0,2024-03-20,38.2


In [2]:
titanic = pd.read_csv(f"{DATA_DIR}/titanic.csv")

titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
round(titanic.isnull().sum()/len(titanic)*100, 1)

PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64

## 2. Null Audit

Before fixing anything, measure the damage.

In [4]:
raw.isnull().mean()

patient_id    0.0000
name          0.0000
age           0.1875
gender        0.0000
bp_sys        0.1875
visit_date    0.0000
temp          0.0625
dtype: float64

In [5]:
(raw.isnull().mean() * 100).round(1)

patient_id     0.0
name           0.0
age           18.8
gender         0.0
bp_sys        18.8
visit_date     0.0
temp           6.2
dtype: float64

In [6]:
raw

,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
1,P002,Sara Khan,28 years,female,118.0,2024-01-11,36.7
2,P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
3,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
4,P004,Zara Malik,45 yr old,FEMALE,132.0,2024-01-13,38.2
5,P005,hassan ALI,67,male,142.0,2024-01-14,34.3
6,P006,Nida Butt,39 years,F,NaN,2024-01-15,38.1
7,P007,USMAN sheikh,NaN,Female,999.0,2024-01-16,35.2
8,P008,Ayesha Nawaz,31 yrs,female,128.0,2024-01-17,36.7
9,P003,BILAL ahmed,51yo,M,158.0,2024-03-20,38.2


In [7]:
raw[~raw.isnull().any(axis=1)]      #returns data with no missing values

,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
1,P002,Sara Khan,28 years,female,118.0,2024-01-11,36.7
3,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
4,P004,Zara Malik,45 yr old,FEMALE,132.0,2024-01-13,38.2
5,P005,hassan ALI,67,male,142.0,2024-01-14,34.3
8,P008,Ayesha Nawaz,31 yrs,female,128.0,2024-01-17,36.7
9,P003,BILAL ahmed,51yo,M,158.0,2024-03-20,38.2
12,P011,Tariq Mahmood,62yo,M,148.0,2024-01-20,35.8
13,P012,Sana Mirza,29 years,female,122.0,2024-01-21,36.8


- `axis=0` --> collapse **rows** --> result has shape `(cols,)` --> one value per column (column wise)
- `axis=1` --> collapse **columns** --> result has shape `(rows,)` --> one value per row (row wise)

In [8]:
rows_with_nulls = raw[raw.isnull().any(axis=1)]     #row-wise
print(f"Rows with at least one missing value: {len(rows_with_nulls)} / {len(raw)}")
rows_with_nulls

Rows with at least one missing value: 7 / 16


,patient_id,name,age,gender,bp_sys,visit_date,temp
2,P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
6,P006,Nida Butt,39 years,F,NaN,2024-01-15,38.1
7,P007,USMAN sheikh,NaN,Female,999.0,2024-01-16,35.2
10,P009,Faisal Iqbal,58 years,male,NaN,2024-01-18,39.2
11,P010,Mariam Syed,NaN,F,135.0,2024-01-19,35.1
14,P013,Ali Hassan,NaN,Male,140.0,2024-01-22,36.9
15,P014,Hina Qureshi,36 yr old,FEMALE,126.0,2024-01-23,NaN


## 3. Handling Missing Values

Two strategies: drop the affected rows, or fill them with a sensible value. The right choice depends on how much data you'd lose and why it's missing - not a default you apply everywhere.

- `dropna()` - remove rows with **ANY** missing value (`default`)
- `dropna(how='all')` - only drop rows where EVERYTHING is missing
- `dropna(subset=...)` - only consider specific columns
- `dropna(thresh=...)` - `keep` rows with `at least N non-null` values

In [9]:
# dropna() - every row with ANY missing value gone. Often too aggressive.
print(f"Rows before: {len(raw)}")
print(f"Rows after dropna(): {len(raw.dropna())}")

Rows before: 16
Rows after dropna(): 9


In [10]:
# dropna(thresh=) - keep rows with at least N non-null values, less aggressive
print(f"thresh=5 (need at least 5 of 6 fields present): {len(raw.dropna(thresh=5))}")

thresh=5 (need at least 5 of 6 fields present): 16


We'll come back to filling values once the columns are actually clean - filling a text age field with a 'mean' makes no sense until it's numeric.

In [11]:
# patients = pd.read_csv(f"{DATA_DIR}/patients.csv")

In [12]:
# # fillna() - replace nulls instead of dropping
# print("Fill with a constant value:")
# print(patients.fillna(0))   # rarely the right choice but shows mechanics

# print("\nFill with column mean (sensible for vitals):")
# filled = patients.copy()
# filled["age"]  = filled["age"].fillna(filled["age"].mean())
# filled["temp"] = filled["temp"].fillna(filled["temp"].mean())
# filled["bp"]   = filled["bp"].fillna(filled["bp"].median())   # median for skew-prone data
# print(filled)

# # fillna with method - forward fill / backward fill (good for time series)
# ts_data = pd.DataFrame({"day": [1,2,3,4,5], "reading": [36.5, np.nan, np.nan, 37.2, 37.0]})
# print(f"\nOriginal:\n{ts_data}")
# print(f"\nForward fill (ffill):\n{ts_data.fillna(method='ffill')}")
# print(f"\nBackward fill (bfill):\n{ts_data.fillna(method='bfill')}")

## 4. Duplicate Detection

In [13]:
raw.duplicated()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
dtype: bool

In [14]:
raw['patient_id'].to_list()

['P001',
 'P002',
 'P003',
 'P001',
 'P004',
 'P005',
 'P006',
 'P007',
 'P008',
 'P003',
 'P009',
 'P010',
 'P011',
 'P012',
 'P013',
 'P014']

In [15]:
raw.duplicated(subset=["patient_id"])       # duplicated(subset=) - only check specific columns for duplication

0     False
1     False
2     False
3      True
4     False
5     False
6     False
7     False
8     False
9      True
10    False
11    False
12    False
13    False
14    False
15    False
dtype: bool

In [16]:
print(f"Exact duplicate rows: {raw.duplicated().sum()}")
print(f"Duplicate patient_ids:  {raw.duplicated(subset=['patient_id']).sum()}")
raw[raw.duplicated(subset=["patient_id"], keep=False)]      #True for duplicate rows (keeps first occurrence as False)

Exact duplicate rows: 0
Duplicate patient_ids:  2


,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
2,P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
3,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
9,P003,BILAL ahmed,51yo,M,158.0,2024-03-20,38.2


P001 is an exact duplicate row - safe to drop. P003 has **two different visits** with different blood pressure readings - dropping the wrong one loses real clinical history. We'll handle these differently below.

## 5. Type Coercion

The `age` column is text (`"34 yrs"`, `"51yo"`, `"NA"`) - that's why a mean or threshold comparison on it would fail right now.

In [17]:
#to check dtpes of full DataFrame (Multiple Columns)
raw.dtypes

patient_id     object
name           object
age            object
gender         object
bp_sys        float64
visit_date     object
temp          float64
dtype: object

In [18]:
print(f"age dtype before cleaning: {raw['age'].dtype}")
raw["age"]

age dtype before cleaning: object


0        34 yrs
1      28 years
2          51yo
3        34 yrs
4     45 yr old
5            67
6      39 years
7           NaN
8        31 yrs
9          51yo
10     58 years
11          NaN
12         62yo
13     29 years
14          NaN
15    36 yr old
Name: age, dtype: object

In [19]:
# Extract the numeric portion with regex, then coerce (force/compel) to numeric
age_numeric = raw["age"].str.extract(r"(\d+)")[0]
age_numeric = pd.to_numeric(age_numeric, errors="coerce")

print(age_numeric)
print(f"\ndtype now: {age_numeric.dtype}")
print(f"Values that failed to extract (now NaN): {age_numeric.isna().sum()}")

0     34.0
1     28.0
2     51.0
3     34.0
4     45.0
5     67.0
6     39.0
7      NaN
8     31.0
9     51.0
10    58.0
11     NaN
12    62.0
13    29.0
14     NaN
15    36.0
Name: 0, dtype: float64

dtype now: float64
Values that failed to extract (now NaN): 3


`errors='coerce'` is the safe default - invalid values become `NaN` instead of crashing the whole pipeline. The alternative, `errors='raise'` (the implicit default), stops on the first bad value.

In [20]:
# # pd.to_datetime() - parsing dates from strings
# dates_df = pd.DataFrame({
#     "patient_id": ["P001","P002","P003","P004"],
#     "admit_date": ["2024-01-15", "15/01/2024", "Jan 15, 2024", "not a date"],
# })

# # Without format specified, Pandas tries to infer — works for most cases
# dates_df["parsed"] = pd.to_datetime(dates_df["admit_date"], errors="coerce")
# print(dates_df)
# print(f"\ndtype: {dates_df['parsed'].dtype}")

# # .astype() - for simple, unambiguous conversions
# scores = pd.DataFrame({"score": [85.7, 92.3, 78.1, 65.9]})
# scores["score_int"] = scores["score"].astype(int)   # truncates, doesn't round!
# print(f"\n{scores}")

# # Common astype targets
# print(f"\nastype examples:")
# print(f"  to int:   {pd.Series([1.0,2.0,3.0]).astype(int).tolist()}")
# print(f"  to float: {pd.Series([1,2,3]).astype(float).tolist()}")
# print(f"  to str:   {pd.Series([1,2,3]).astype(str).tolist()}")
# print(f"  to category: {pd.Series(['A','B','A','C']).astype('category').dtype}")

## 6. String Cleanup

Vectorised `.str` methods - the Module 1 string toolkit applied to an entire column at once.

In [21]:
raw["name"]

0         ahmad RAZA
1          Sara Khan
2        BILAL ahmed
3         ahmad RAZA
4       Zara  Malik 
5         hassan ALI
6          Nida Butt
7       USMAN sheikh
8       Ayesha Nawaz
9        BILAL ahmed
10      Faisal Iqbal
11       Mariam Syed
12     Tariq Mahmood
13        Sana Mirza
14        Ali Hassan
15      Hina Qureshi
Name: name, dtype: object

In [22]:
# raw['name'].str.strip().str.split()

In [24]:
# .strip() only removes LEADING/TRAILING whitespace - not internal double-spaces
name_clean = raw["name"].str.strip().str.split().str.join(" ").str.title()
name_clean

0        Ahmad Raza
1         Sara Khan
2       Bilal Ahmed
3        Ahmad Raza
4        Zara Malik
5        Hassan Ali
6         Nida Butt
7      Usman Sheikh
8      Ayesha Nawaz
9       Bilal Ahmed
10     Faisal Iqbal
11      Mariam Syed
12    Tariq Mahmood
13       Sana Mirza
14       Ali Hassan
15     Hina Qureshi
Name: name, dtype: object

In [25]:
raw["gender"].unique()

array(['Male', 'female ', 'M', 'FEMALE', 'male', 'F', 'Female', 'female'],
      dtype=object)

In [26]:
gender_clean = (raw["gender"].str.strip().str.lower()
                .replace({"male": "M", "m": "M", "female": "F", "f": "F"}))
gender_clean.value_counts()

gender
M    8
F    8
Name: count, dtype: int64

## 7. Putting It Together

Assemble every fix into one cleaned DataFrame, in a deliberate order: clean text first, coerce types, handle the BP outlier, fill what's left, then deduplicate last (since deduplication depends on having clean comparable values).

In [27]:
df = raw.copy()

df["name"]   = df["name"].str.strip().str.split().str.join(" ").str.title()
df["gender"] = (df["gender"].str.strip().str.lower()
                .replace({"male":"M","m":"M","female":"F","f":"F"}))
df["age"]    = pd.to_numeric(df["age"].str.extract(r"(\d+)")[0], errors="coerce")

df

,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,Ahmad Raza,34.0,M,145.0,2024-01-10,38.9
1,P002,Sara Khan,28.0,F,118.0,2024-01-11,36.7
2,P003,Bilal Ahmed,51.0,M,NaN,2024-01-12,39.4
3,P001,Ahmad Raza,34.0,M,145.0,2024-01-10,37.1
4,P004,Zara Malik,45.0,F,132.0,2024-01-13,38.2
5,P005,Hassan Ali,67.0,M,142.0,2024-01-14,34.3
6,P006,Nida Butt,39.0,F,NaN,2024-01-15,38.1
7,P007,Usman Sheikh,NaN,F,999.0,2024-01-16,35.2
8,P008,Ayesha Nawaz,31.0,F,128.0,2024-01-17,36.7
9,P003,Bilal Ahmed,51.0,M,158.0,2024-03-20,38.2


## The 999 outlier

A systolic BP of 999 is not real data - it's almost certainly a sentinel value from a broken sensor or a data-entry placeholder. **Scan for impossible values before computing any fill statistic** - a single 999 will drag every imputed mean off target.

In [ ]:
#do you know the any other way to filter systolic bp below 300???


136.58333333333334

In [32]:
print(f"Mean BP WITH the 999 included: {df['bp_sys'].mean():.1f}")

df["bp_sys"] = df["bp_sys"].where(df["bp_sys"] < 300)   # blank out the impossible value
correct_mean = df["bp_sys"].mean()
print(f"Mean BP WITHOUT the 999:           {correct_mean:.1f}")

df["bp_sys"] = df["bp_sys"].fillna(correct_mean)
df[["patient_id","bp_sys"]]

Mean BP WITH the 999 included: 202.9
Mean BP WITHOUT the 999:           136.6


,patient_id,bp_sys
0,P001,145.000000
1,P002,118.000000
2,P003,136.583333
3,P001,145.000000
4,P004,132.000000
5,P005,142.000000
6,P006,136.583333
7,P007,136.583333
8,P008,128.000000
9,P003,158.000000


## 8. Deduplication - with intent

P001 is an exact duplicate - drop it outright. P003 has two genuine visits - sort by date and keep the most recent, preserving the latest clinical status rather than arbitrarily keeping whichever row happened to load first.

In [38]:
df = df.sort_values("visit_date")
df_final = df.drop_duplicates(subset=["patient_id"], keep="last")       #remove exact duplicate rows, keeps first by default
df_final = df_final.sort_values("patient_id").reset_index(drop=True)
df_final

,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,Ahmad Raza,34.0,M,145.000000,2024-01-10,37.1
1,P002,Sara Khan,28.0,F,118.000000,2024-01-11,36.7
2,P003,Bilal Ahmed,51.0,M,158.000000,2024-03-20,38.2
3,P004,Zara Malik,45.0,F,132.000000,2024-01-13,38.2
4,P005,Hassan Ali,67.0,M,142.000000,2024-01-14,34.3
5,P006,Nida Butt,39.0,F,136.583333,2024-01-15,38.1
6,P007,Usman Sheikh,NaN,F,136.583333,2024-01-16,35.2
7,P008,Ayesha Nawaz,31.0,F,128.000000,2024-01-17,36.7
8,P009,Faisal Iqbal,58.0,M,136.583333,2024-01-18,39.2
9,P010,Mariam Syed,NaN,F,135.000000,2024-01-19,35.1


When resetting row index labels (e.g., after sorting or filtering), Pandas moves the old index into a new column named 'index' by default. Passing drop=True discards the old index entirely instead of creating that extra column.

## 9. Column renaming and standardisation

In [ ]:
# df = pd.DataFrame({
#     "Patient ID":     ["P001","P002","P003"],
#     "  First Name  ": ["Ahmad","Sara","Bilal"],
#     "BP (Systolic)":  [145, 118, 158],
# })
# print(f"Original columns: {list(df.columns)}")

# # .rename() - explicit mapping for specific columns
# renamed = df.rename(columns={
#     "Patient ID": "patient_id",
#     "  First Name  ": "first_name",
#     "BP (Systolic)": "bp_systolic"
# })
# print(f"\nAfter explicit rename: {list(renamed.columns)}")

# # Bulk standardisation - the production pattern you'll use constantly
# df.columns = (df.columns
#               .str.strip()
#               .str.lower()
#               .str.replace(" ", "_")
#               .str.replace(r"[()]", "", regex=True))
# print(f"\nAfter bulk standardisation: {list(df.columns)}")
# print(df)

Original columns: ['Patient ID', '  First Name  ', 'BP (Systolic)']

After explicit rename: ['patient_id', 'first_name', 'bp_systolic']

After bulk standardisation: ['patient_id', 'first_name', 'bp_systolic']
  patient_id first_name  bp_systolic
0       P001      Ahmad          145
1       P002       Sara          118
2       P003      Bilal          158


## Practical

### Practical 1 - fillna(mean) before vs after removing outliers

In [66]:
naive_mean = raw["bp_sys"].mean()
correct_mean = raw["bp_sys"].where(raw["bp_sys"] < 300).mean()

print(f"Naive mean (999 included):    {naive_mean:.1f}")
print(f"Correct mean (999 excluded):  {correct_mean:.1f}")
print(f"Difference: {naive_mean - correct_mean:.1f} mmHg")
print()
print("A single impossible value shifted the imputed BP for every missing patient")
print("by a wide margin. ALWAYS scan for impossible values before filling.")

Naive mean (999 included):    202.9
Correct mean (999 excluded):  136.6
Difference: 66.3 mmHg

A single impossible value shifted the imputed BP for every missing patient
by a wide margin. ALWAYS scan for impossible values before filling.


### Pratcical 2 - drop_duplicates(keep=) changes your dataset silently

In [67]:
p003 = raw[raw["patient_id"] == "P003"]
print("P003's two visits:")
print(p003[["patient_id","bp_sys","visit_date"]])

naive = raw.drop_duplicates(subset=["patient_id"], keep="first")
print(f"\nkeep='first' (DEFAULT) keeps bp_sys={naive[naive['patient_id']=='P003']['bp_sys'].values}")
print("  <- this is the row with a MISSING bp_sys, not the follow-up reading!")

sorted_then_last = raw.sort_values("visit_date").drop_duplicates(subset=["patient_id"], keep="last")
kept = sorted_then_last[sorted_then_last["patient_id"]=="P003"]["bp_sys"].values
print(f"\nSorted by date, keep='last' keeps bp_sys={kept}  <- the actual follow-up reading")
print()
print("'first'/'last' depend entirely on row ORDER. Sort meaningfully before deduplicating.")

P003's two visits:
  patient_id  bp_sys  visit_date
2       P003     NaN  2024-01-12
9       P003   158.0  2024-03-20

keep='first' (DEFAULT) keeps bp_sys=[nan]
  <- this is the row with a MISSING bp_sys, not the follow-up reading!

Sorted by date, keep='last' keeps bp_sys=[158.]  <- the actual follow-up reading

'first'/'last' depend entirely on row ORDER. Sort meaningfully before deduplicating.


### Pratcical 3 - errors='coerce' vs the default

In [68]:
print("Default behaviour (errors='raise', implicit):")
try:
    pd.to_numeric(raw["age"])
except ValueError as e:
    print(f"  CRASHES: {e}")
    print("  The entire pipeline stops on ONE bad value.")

print("\nerrors='coerce':")
result = pd.to_numeric(raw["age"].str.extract(r"(\d+)")[0], errors="coerce")
print(f"  {result.tolist()}")
print(f"  Bad/missing values became NaN. Pipeline survives.")
print(f"  You can now report exactly which rows failed:")
print(raw[result.isna()][["patient_id","age"]])

Default behaviour (errors='raise', implicit):
  CRASHES: Unable to parse string "34 yrs" at position 0
  The entire pipeline stops on ONE bad value.

errors='coerce':
  [34.0, 28.0, 51.0, 34.0, 45.0, 67.0, 39.0, nan, 31.0, 51.0, 58.0, nan, 62.0, 29.0, nan, 36.0]
  Bad/missing values became NaN. Pipeline survives.
  You can now report exactly which rows failed:
   patient_id  age
7        P007  NaN
11       P010  NaN
14       P013  NaN


### Practical 4 - .str.strip() doesn't fix internal whitespace

In [70]:
original = raw["name"].iloc[4]   # ' Zara  Malik '
print(f"Original:        {original!r}")
print(f"After .strip():  {original.strip()!r}   <- internal double-space survives")
print(f"After split/join:{' '.join(original.split())!r}   <- actually fixed")
print()
print("Always use .str.strip().str.split().str.join(' ') for full whitespace")
print("normalisation, not .str.strip() alone - internal spacing breaks merges later.")

Original:        ' Zara  Malik '
After .strip():  'Zara  Malik'   <- internal double-space survives
After split/join:'Zara Malik'   <- actually fixed

Always use .str.strip().str.split().str.join(' ') for full whitespace
normalisation, not .str.strip() alone - internal spacing breaks merges later.


## Validation Report

The final, formal before/after comparison - every cleaning pipeline should end with one of these.

In [71]:
before = {
    "rows":        len(raw),
    "nulls":       int(raw.isnull().sum().sum()),
    "duplicates":  int(raw.duplicated(subset=["patient_id"]).sum()),
}
after = {
    "rows":        len(df_final),
    "nulls":       int(df_final[["age","gender","bp_sys"]].isnull().sum().sum()),
    "duplicates":  int(df_final.duplicated(subset=["patient_id"]).sum()),
}

report = pd.DataFrame({"Before": before, "After": after})
report

,Before,After
rows,16,14
nulls,7,3
duplicates,2,0


## Summary

| Concept | Key point |
|---|---|
| `.isnull().sum()` | Always quantify missingness before deciding how to handle it |
| `.dropna(thresh=)` | Keep rows with at least N non-null values - gentler than `how='any'` |
| Outliers before fillna | Scan for impossible values FIRST - one extreme value corrupts the mean |
| `errors='coerce'` | Safe default for `to_numeric`/`to_datetime` - bad values become NaN |
| `drop_duplicates(keep=)` | Sort meaningfully BEFORE deduplicating - order determines the outcome |
| `.str.strip()` | Only fixes leading/trailing whitespace - use `.split().join(' ')` for internal spacing |
| `.str.extract(regex)` | Pulls structured values out of messy text columns |
| Column renaming | Standardise to `snake_case` immediately after loading — saves friction later |

## Homework

**Module 3 is half complete.** Week 4 covered loading, selecting, and cleaning.
Week 5 covers `groupby`, merging, reshaping, and time series - review this
notebook's cleaning pipeline once more; Week 5 builds directly on a properly
cleaned DataFrame.

Questions:

1. In a DataFrame with 7 columns, what does df.dropna(thresh=5) do? How is it safer than default df.dropna() when working with incomplete clinical records?
2. Why does running df['age'].astype(int) throw an error even after extracting digits, and why must age remain a float64 or Int64 (nullable integer) type?
3. Why must you perform string normalization and type coercion BEFORE running deduplication (.drop_duplicates())?
4. A student writes df.dropna(inplace=True) and df.drop_duplicates(inplace=True) throughout their cleaning pipeline. Why is inplace=True increasingly discouraged in modern Pandas production code in favor of explicit reassignments like df = df.dropna()?
5. 